<a href="https://colab.research.google.com/github/ruveydakarakoyun/APTOS-2019/blob/main/aptos_2019.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Veri Yükleme ve Kütüphaneleri Yükleme

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU kullanılabilir mi?:", torch.cuda.is_available())

### Google Cloud kimlik dogrulama

In [ ]:
from google.colab import auth

auth.authenticate_user()
print("Kimlik dogrulandi.")

### Yapilandirma

In [ ]:
# ==============================================================================
# PROJE YAPILANDIRMASI - tek kaynak, asagidaki hucreler bunlari kullanir
# ==============================================================================
PROJECT_ID  = "datascientis"
BUCKET_NAME = "aptos2019-retina-images"
BQ_DATASET  = "APTOS_2019"

LOCAL_DIR = "/content/aptos_images"   # goruntulerin yerel onbellegi

BQ_TABLES = {
    "train": f"{PROJECT_ID}.{BQ_DATASET}.aptos_train",
    "valid": f"{PROJECT_ID}.{BQ_DATASET}.aptos_valid",
    "test":  f"{PROJECT_ID}.{BQ_DATASET}.aptos_test",
}


def blob_name(image_uri):
    """gs://bucket/klasor/dosya.png  ->  klasor/dosya.png"""
    return image_uri.replace(f"gs://{BUCKET_NAME}/", "", 1)


def local_path(image_uri):
    return f"{LOCAL_DIR}/{blob_name(image_uri)}"


print("Yapilandirma hazir.")

###BigQuery'den Tabloları Çekme

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=PROJECT_ID)


# image_uri kolonu tam gs:// yolunu tasir. Klasor prefix'i (aptos_train_images/ gibi)
# bu kolonun icinde oldugu icin dosya yolunu elle kurmaya gerek yok.
def fetch(split):
    q = f"SELECT id_code, diagnosis, image_file, image_uri FROM `{BQ_TABLES[split]}`"
    return bq_client.query(q).to_dataframe()


df_train, df_valid, df_test = fetch("train"), fetch("valid"), fetch("test")

for name, df in [("Train", df_train), ("Valid", df_valid), ("Test", df_test)]:
    print(f"{name:<6} {len(df):>5} satir")

print("\nornek image_uri:", df_train["image_uri"].iloc[0])

### Goruntuleri yerel diske indirme

In [ ]:
# ==============================================================================
# GORUNTULERI YEREL DISKE INDIR  (bir kez calisir, ~8 GB)
# ==============================================================================
# Neden: her __getitem__ cagrisinda GCS'ten tek tek indirmek ~2.7 goruntu/saniye
# demek; 2930 goruntuluk bir epoch sadece okuma icin ~18 dakika surer.
# Yerel diske bir kez kopyalayinca egitim I/O'ya takilmaz.
import os

os.makedirs(LOCAL_DIR, exist_ok=True)
!gsutil -m -q cp -r "gs://$BUCKET_NAME/*" "$LOCAL_DIR/"

for split, df in [("train", df_train), ("valid", df_valid), ("test", df_test)]:
    have = sum(os.path.exists(local_path(u)) for u in df["image_uri"])
    print(f"{split:<6} {have}/{len(df)} goruntu yerelde")

### Yollarin dogrulanmasi

In [ ]:
# image_uri yollarinin gercekten bucket'ta oldugunu dogrula
from google.cloud import storage

bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

for uri in df_train["image_uri"].head(3):
    durum = "BULUNDU" if bucket.blob(blob_name(uri)).exists() else "YOK"
    print(f"{uri}\n   -> {durum}")

### Dataset sinifi ve auto-crop

In [ ]:
import io
import os

import cv2
import numpy as np
import torch
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset


def crop_image_from_gray(img, tol=7):
    """Retina etrafindaki siyah boslugu keser."""
    if img.ndim == 2:
        mask = img > tol
        return img[np.ix_(mask.any(1), mask.any(0))]

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if img[:, :, 0][np.ix_(mask.any(1), mask.any(0))].shape[0] == 0:
        return img
    return np.stack(
        [img[:, :, i][np.ix_(mask.any(1), mask.any(0))] for i in range(3)], axis=-1
    )


class APTOSDataset(Dataset):
    """Goruntuleri yerel onbellekten okur; dosya yoksa GCS'ten ceker."""

    def __init__(self, df, transform=None, use_crop=True):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.use_crop = use_crop
        self._bucket = None   # GCS'e ancak gerekirse baglanir

    def __len__(self):
        return len(self.df)

    def read_bytes(self, image_uri):
        path = local_path(image_uri)
        if os.path.exists(path):
            with open(path, "rb") as fh:
                return fh.read()

        if self._bucket is None:                       # yedek yol
            from google.cloud import storage
            self._bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
        return self._bucket.blob(blob_name(image_uri)).download_as_bytes()

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(io.BytesIO(self.read_bytes(row["image_uri"]))).convert("RGB")

        if self.use_crop:
            img = Image.fromarray(crop_image_from_gray(np.array(img)))

        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(int(row["diagnosis"]), dtype=torch.long)

###Dataframe'leri ve Transform'ları Ayırma

In [ ]:
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = APTOSDataset(df_train, transform=train_transform)
valid_dataset = APTOSDataset(df_valid, transform=eval_transform)
test_dataset  = APTOSDataset(df_test,  transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False,
                          num_workers=2, pin_memory=True)

imgs, labels = next(iter(train_loader))
print("batch:", imgs.shape, labels.shape)

#EDA

###Sınıf dağılımı (Evre 0 - Evre 4 kaçar adet?) grafikleri ve Sınıf dengesizliği oranları

In [ ]:
# Sınıf dağılımı
# Sınıf isimleri
class_names = {
    0: "No DR",
    1: "Mild",
    2: "Moderate",
    3: "Severe",
    4: "Proliferative DR"
}

datasets = {
    "Train": df_train,
    "Validation": df_valid,
    "Test": df_test
}

for dataset_name, df in datasets.items():

    # Her sınıftaki görüntü sayısı
    counts = (
        df["diagnosis"]
        .value_counts()
        .reindex(range(5), fill_value=0)
        .sort_index()
    )

    # Yüzdelik dağılım
    percentages = (counts / counts.sum()) * 100

    # Sonuç tablosu
    distribution_table = pd.DataFrame({
        "Evre": counts.index,
        "Sınıf": [class_names[i] for i in counts.index],
        "Görüntü Sayısı": counts.values,
        "Yüzde (%)": percentages.round(2).values
    })

    print(f"\n📊 {dataset_name} Sınıf Dağılımı")
    display(distribution_table)

    # Sınıf dengesizliği
    non_zero_counts = counts[counts > 0]

    max_count = non_zero_counts.max()
    min_count = non_zero_counts.min()

    imbalance_ratio = max_count / min_count

    print(f"En fazla görüntü: Evre {non_zero_counts.idxmax()} → {max_count}")
    print(f"En az görüntü: Evre {non_zero_counts.idxmin()} → {min_count}")
    print(f"Sınıf dengesizlik oranı (Max / Min): {imbalance_ratio:.2f}")

    # Grafik
    plt.figure(figsize=(8, 5))

    bars = plt.bar(
        [f"Evre {i}" for i in counts.index],
        counts.values
    )

    plt.title(f"{dataset_name} - Sınıf Dağılımı")
    plt.xlabel("Diyabetik Retinopati Evresi")
    plt.ylabel("Görüntü Sayısı")

    for bar, count, percentage in zip(
        bars, counts.values, percentages.values
    ):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{count}\n%{percentage:.1f}",
            ha="center",
            va="bottom"
        )

    plt.tight_layout()
    plt.show()

###Veri Temizleme ve Kalite Kontrol Betiği

In [ ]:
from PIL import Image, ImageStat
from tqdm.auto import tqdm


def check_dataset_quality(df, brightness_threshold=12.0):
    """Bozuk/acilmayan ve asiri karanlik goruntuleri bulur."""
    reader = APTOSDataset(df, use_crop=False)
    corrupted, too_dark, saglam = [], [], []

    for idx in tqdm(range(len(df)), desc="Goruntuler kontrol ediliyor"):
        row = df.iloc[idx]
        try:
            img = Image.open(io.BytesIO(reader.read_bytes(row["image_uri"]))).convert("RGB")
            brightness = ImageStat.Stat(img.convert("L")).mean[0]
            hedef = too_dark if brightness < brightness_threshold else saglam
            hedef.append({"id_code": row["id_code"], "brightness": brightness})
        except Exception as e:
            corrupted.append({"id_code": row["id_code"], "error": str(e)})

    print("\n" + "=" * 40)
    print("KONTROL SONUCLARI")
    print("=" * 40)
    print(f"Saglam goruntu    : {len(saglam)}")
    print(f"Bozuk / acilmayan : {len(corrupted)}")
    print(f"Asiri karanlik    : {len(too_dark)}")
    return saglam, corrupted, too_dark


saglam_list, corrupted_list, dark_list = check_dataset_quality(df_train)